In [33]:
import re
from urllib.parse import unquote
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [42]:
# --- Helper Functions ---
def scraper(url: str, driver, event, genre) -> pd.DataFrame:
    """
    Takes the url of a web page, scrapes all relevant information, and put them
    in a dataframe.
    """
    driver.get(url)

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "tbody"))
        )

        records = []
    
        tbody = driver.find_element(By.TAG_NAME, "tbody")
        rows = tbody.find_elements(By.TAG_NAME, "tr")

        for row in rows:
            seen = set()
            candidate_entries = []
            cells = row.find_elements(By.TAG_NAME, "td")
            song = cells[0].text.strip()
            performer = cells[1].text.strip()
            candidates_cell = cells[2]
            event_title = cells[3].text.strip()
            date = cells[4].text.strip()

            spans = candidates_cell.find_elements(By.XPATH, ".//span")

            for span in spans:
                try:
                    svg = span.find_element(By.TAG_NAME, "svg")
                    svg_class = svg.get_attribute("class") or ""
                    if "fa-thumbs-up" in svg_class:
                        sentiment = "thumbs-up"
                    elif "fa-thumbs-down" in svg_class:
                        sentiment = "thumbs-down"
                    elif "fa-question-circle" in svg_class:
                        sentiment = "question-circle"
                    else:
                        continue
                except:
                    continue

                try:
                    party = span.find_element(By.TAG_NAME, "sup").text.strip()
                except:
                    party = ""

                full_text = span.text.strip()
                if party and full_text.endswith(party):
                    name = full_text[: -len(party)].strip()
                else:
                    name = full_text

                candidate_full = f"{sentiment} {name} {party}".strip()
                if candidate_full not in seen:
                    seen.add(candidate_full)
                    candidate_entries.append(candidate_full)

            candidates = ', '.join(candidate_entries)

            records.append({
                "Song": song,
                "Performer": performer,
                "Candidates": candidates,
                "Event Title": event_title,
                "Date": date,
                "Event Type": unquote(event),
                "Genre": unquote(genre)
            })

        df = pd.DataFrame(records)
        return df

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return pd.DataFrame(columns=["Song", "Performer", "Candidates", "Event Title", 
                                     "Date", "Event Type", "Genre"])
    

def wait_for_count(driver, STATS_SEL, HITS_SEL, timeout=20):
    """
    Wait for the stats box to appear and return the number of results for a given
    category. If the number is 0, confirm with presence of hit cards.
    """
    WebDriverWait(driver, timeout).until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, STATS_SEL))
    )

    def _parse_count():
        """
        Get the total number of results.
        """
        txt = driver.find_element(By.CSS_SELECTOR, STATS_SEL).text.strip()
        m = re.search(r"(\d+)", txt)
        return int(m.group(1)) if m else None

    count = WebDriverWait(driver, timeout).until(lambda d: _parse_count() is not None or False)
    count = _parse_count()

    if count == 0:
        time.sleep(0.5)
        txt = driver.find_element(By.CSS_SELECTOR, STATS_SEL).text.strip()
        m = re.search(r"(\d+)", txt)
        if m:
            count = int(m.group(1))

        if count == 0:
            hits = driver.find_elements(By.CSS_SELECTOR, HITS_SEL)
            if len(hits) > 0:
                count = len(hits)

    return count

In [36]:
# --- Selenium ---
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)

# --- Parameters ---
lst_genre = ["pop%2Frock", "country", "patriotic", "religious", "classical", "r%26b", 
             "latin", "rap", "electronic", "jazz", "traditional", "folk", "reggae", 
             "stage%20%26%20screen", "vocal"]
lst_year = ["2024", "2020", "2016"]
lst_event = ["campaign%20rally", "online%20media", "convention", "other", 
             "campaign%20launch", "playlist", "concert", "fundraising%20concert", 
             "fundraiser", "speech", "town%20hall%20meeting", "advertisement", 
             "campaign%20stop", "late%20show", "organization%20video", 
             "daytime%20talk%20show", "protest", "online%20news", "", "cd", 
             "mock%20or%20parody%20advertisement"]
base_url = (
    "https://db.traxonthetrail.com/?Genre[0]={}"
    "&Type[0]=preexisting"
    "&Campaign[0]={}%20Campaign"
    "&Event_Type[0]={}"
)
STATS_SEL = "div.sk-hits-stats__info[data-qa='info']"
HITS_SEL = ".sk-hits-list .sk-hits-hit"

# --- Scraping ---
lst = []
try:
    for year in lst_year:
        for event in lst_event:
            for genre in lst_genre:
                base = base_url.format(genre, year, event)
                driver.get(base)

                num_results = wait_for_count(driver, STATS_SEL, HITS_SEL, timeout=20)
                print("Stats text:", driver.find_element(By.CSS_SELECTOR, STATS_SEL).text.strip())
                print("Parsed num_results:", num_results)

                if not num_results or num_results == 0:
                    print(f"No results for year={year}, event={event}, genre={genre}")
                    continue

                num_pages = (num_results + 19) // 20
                for n in range(1, num_pages + 1):
                    page_url = f"{base}&p={n}"
                    new = scraper(page_url, driver, event, genre)
                    if isinstance(new, pd.DataFrame) and not new.empty:
                        lst.append(new)
                    print(f"Page {n} scraped for campaign {year}, {unquote(event)}, and {unquote(genre)}")

    df = pd.concat(lst, ignore_index=True) if lst else pd.DataFrame(
        columns=["Song", "Performer", "Candidates", "Event Title", "Date", 
                 "Event Type", "Genre"]
    )
    print(f"Scraping completed. Total records collected: {len(df)}")

finally:
    driver.quit()

Stats text: 260 results found
Parsed num_results: 260
Page 1 scraped for campaign 2024, campaign rally, and pop/rock
Page 2 scraped for campaign 2024, campaign rally, and pop/rock
Page 3 scraped for campaign 2024, campaign rally, and pop/rock
Page 4 scraped for campaign 2024, campaign rally, and pop/rock
Page 5 scraped for campaign 2024, campaign rally, and pop/rock
Page 6 scraped for campaign 2024, campaign rally, and pop/rock
Page 7 scraped for campaign 2024, campaign rally, and pop/rock
Page 8 scraped for campaign 2024, campaign rally, and pop/rock
Page 9 scraped for campaign 2024, campaign rally, and pop/rock
Page 10 scraped for campaign 2024, campaign rally, and pop/rock
Page 11 scraped for campaign 2024, campaign rally, and pop/rock
Page 12 scraped for campaign 2024, campaign rally, and pop/rock
Page 13 scraped for campaign 2024, campaign rally, and pop/rock
Stats text: 82 results found
Parsed num_results: 82
Page 1 scraped for campaign 2024, campaign rally, and country
Page 2 sc

In [40]:
df

,Song,Performer,Candidates,Event Title,Date,Event Type,Genre
0,Still the One,Orleans,thumbs-up Donald Trump R,,2024-11-03,campaign rally,pop/rock
1,Hit Me with Your Best Shot,Pat Benatar,thumbs-up Donald Trump R,,2024-11-03,campaign rally,pop/rock
2,Bod Moon Rising,Credence Clearwater Revival,thumbs-up Donald Trump R,,2024-11-03,campaign rally,pop/rock
3,You're the Best,Joe Esposito,thumbs-up Donald Trump R,,2024-11-03,campaign rally,pop/rock
4,Don't Stop the Music,Rihanna,thumbs-up Donald Trump R,,2024-11-03,campaign rally,pop/rock
...,...,...,...,...,...,...,...
7302,Watch Me (Whip/Nae Nae),Silentó,question-circle Hillary Clinton D,The Ellen Degeneres Show,2015-09-08,daytime talk show,rap
7303,America,Simon & Garfunkel,"question-circle Bernie Sanders D, question-cir...","Bernie Sanders ""The View"" - Watch Full Interview!",2016-04-08,daytime talk show,folk
7304,Liar,Henry Rollins,question-circle Hillary Clinton D,Hillary Clinton Theme Song (Henry Rollins' Liar),2016-08-05,online news,pop/rock
7305,Rebel Girl,Bikini Kill,question-circle Hillary Clinton D,,2015-09-01,online news,pop/rock


In [41]:
df.to_csv('scraped_data_new.csv', index=False)